# Convert Data Structures to Cell Format

This notebook demonstrates converting various Python data structures (lists, dictionaries, pandas DataFrames) into "cell"-style arrays for efficient storage and manipulation, and exporting them to common formats like `.mat`, `.csv`, and `.xlsx`.

Notes:
- "Cell format" here is represented using nested Python lists (list-of-lists) similar to MATLAB cell arrays.
- Install required packages with `pip install pandas numpy scipy openpyxl` if needed.

In [ ]:
# 1) Import required libraries
import numpy as np
import pandas as pd
import json
from scipy import io as scipy_io

print("numpy", np.__version__, "pandas", pd.__version__)


In [ ]:
# 2) Load or create sample data
# Python list and nested list
simple_list = [1, 2, 3, 4]
nested_list = [[1, 2], [3, 4], [5, 6]]

# Dictionary with mixed types
sample_dict = {"name": "Alice", "age": 30, "scores": [90, 85, 92]}

# Pandas DataFrame with mixed dtype and missing values
df = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", None],
    "value": [3.14, np.nan, 2.72]
})

print("sample_list:", simple_list)
print("nested_list:", nested_list)
print("sample_dict:", sample_dict)
print("df:\n", df)


In [ ]:
# 3) Convert Lists to cell arrays

def list_to_cell(lst):
    """Convert a flat list to a cell-style list-of-lists (each element becomes a single-item cell)."""
    return [[x] for x in lst]


def nested_list_to_cell(nested):
    """Convert nested lists to cell (preserve sublists)."""
    return [list(item) for item in nested]

cell_from_simple = list_to_cell(simple_list)
cell_from_nested = nested_list_to_cell(nested_list)

print("cell_from_simple:", cell_from_simple)
print("cell_from_nested:", cell_from_nested)


In [ ]:
# 4) Convert Dictionaries to cell format

def dict_to_cell(d: dict):
    """Convert a dict to a cell-style list of [key, value] pairs."""
    return [[k, d[k]] for k in d]

cell_from_dict = dict_to_cell(sample_dict)
print("cell_from_dict:", cell_from_dict)

# Alternatively, convert dict of lists into a table (rows)
if all(isinstance(v, list) for v in sample_dict.values()):
    # Align by index
    keys = list(sample_dict.keys())
    rows = []
    for i in range(len(next(iter(sample_dict.values())))):
        rows.append([sample_dict[k][i] for k in keys])
    print("dict-as-rows:", rows)


In [ ]:
# 5) Convert DataFrames to cell arrays

# Convert DataFrame to a list-of-lists (cells) including header

def df_to_cell(df: pd.DataFrame, include_header=True):
    header = list(df.columns) if include_header else []
    rows = df.where(pd.notnull(df), None).values.tolist()  # convert NaN -> None for portability
    if include_header:
        return [header] + rows
    return rows

cell_from_df = df_to_cell(df)
print("cell_from_df:")
for r in cell_from_df:
    print(r)


In [ ]:
# 6) Access and manipulate cell data

# Example: access first data row (after header)
if len(cell_from_df) > 1:
    first_row = cell_from_df[1]
    print("first_row:", first_row)

# Modify a cell value
modified = [row.copy() for row in cell_from_df]
if len(modified) > 1:
    modified[1][2] = 999  # set column index 2 of first data row
    print("modified first data row:", modified[1])

# Slice: get column 0 values from data rows
col0 = [row[0] for row in modified[1:]]
print("column 0 values:", col0)


In [ ]:
# 7) Export cell format data

# Export DataFrame-derived cell to CSV
import csv
with open('cell_from_df.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    for row in cell_from_df:
        writer.writerow([json.dumps(x) if isinstance(x, (list, dict)) else x for x in row])
print("Wrote cell_from_df.csv")

# Export to Excel (requires openpyxl)
try:
    pd.DataFrame(cell_from_df[1:], columns=cell_from_df[0]).to_excel('cell_from_df.xlsx', index=False)
    print("Wrote cell_from_df.xlsx")
except Exception as e:
    print("Could not write Excel:", e)

# Export to MATLAB .mat file (scipy.io)
scipy_io.savemat('cell_from_df.mat', {'cell': cell_from_df})
print("Wrote cell_from_df.mat")


## Notes and next steps

- The notebook shows straightforward conversions between Python data types and a "cell" representation using nested lists.
- For large datasets consider using NumPy structured arrays, HDF5, or Parquet for efficiency instead of storing everything as nested Python lists.
- When exporting to `.mat`, note that complex nested Python objects may not round-trip perfectly; consider converting to lists/arrays or using JSON for fidelity.
